<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/Kalman_Filter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports and Setup

In [ ]:
import numpy as np
import scipy as sp
import pandas as pd
from scipy.integrate import odeint
import math
from numpy import linalg
import sympy
from sympy import symbols
from sympy import *

import plotly.graph_objects as go
import plotly.express as px
from sympy.physics.mechanics import dynamicsymbols, init_vprinting
from IPython.display import display, Math, Latex

In [ ]:
!pip install --quiet "git+https://github.com/mugalan/classical-mechanics-from-a-geometric-point-of-view.git#egg=rigid-body-sim"
import sims
mr = sims.RigidBodySim()

# The Kalman Filter on $\mathbb{R}^n$

Consider the linear Gaussian process:
\begin{align*}
x_k &= A_{k-1}\,x_{k-1} + G_{k-1}\,w_{k-1}, \\
y_k &= H_k\,x_k + z_k,
\end{align*}
where $w_k \sim \mathscr{N}(0,\Sigma_p)$ and $z_k \sim \mathscr{N}(0,\Sigma_m)$ are mutually independent white noise sequences, also independent of the initial state $x_0 \sim \mathscr{N}(m_0, P_0)$.
Here $y_k$ denotes the random variable representing the measurement at time step $k$, while $y_k^{\mathrm{obs}}$ denotes its observed numerical realization.

---

**Prediction Step (Time Update):**

Define the filter model
\begin{align*}
x^{-}_k &= A_{k-1}\,x^{+}_{k-1} + G_{k-1}\,w_{k-1}, \\
y^{-}_k &= H_k\,x^{-}_k + z_k,
\end{align*}
where
$$x^{+}_{k-1} \sim \mathscr{N}(m_{k-1}, P_{k-1}),$$


Because the filter equation is linear and the process noise is Gaussian, the predicted state $x^{-}_k$ is also Gaussian. Taking the expectation of the state equation yields the predicted mean:


$$m_k^- \triangleq \mathbb{E}[x^{-}_k] = A_{k-1}\mathbb{E}[x^{+}_{k-1}] + G_{k-1}\mathbb{E}[w_{k-1}] = A_{k-1}m_{k-1},$$


since $\mathbb{E}[w_{k-1}] = 0$.

The predicted covariance $P_k^-$ is computed by applying the variance operator to the state equation:
\begin{align*}
P_k^- &\triangleq \text{Var}(x^{-}_k) \\
&= A_{k-1}\text{Var}(x^{+}_{k-1})A_{k-1}^T + G_{k-1}\text{Var}(w_{k-1})G_{k-1}^T \\
&= A_{k-1}P_{k-1}A_{k-1}^T + G_{k-1}\Sigma_p G_{k-1}^T.
\end{align*}
Thus, prior to incorporating the new measurement, our belief of the state is characterized by the prior distribution:


$$x_k^- \sim \mathscr{N}(m_k^-, P_k^-).$$

---

**Measurement Prediction:**

Prior to its physical realization, the upcoming measurement at time step $k$ is treated as the random variable $y_k$. Based on our prior state belief, its expected value is:


$$\mathbb{E}[y^{-}_k] = H_k \mathbb{E}[x_k^-] + \mathbb{E}[z_k] = H_k m_k^-,$$


and its variance is:


$$\text{Var}(y^{-}_k) = H_k \text{Var}(x_k^-) H_k^T + \text{Var}(z_k) = H_k P_k^- H_k^T + \Sigma_m.$$

---

**Joint Prior Distribution:**

The cross-covariance between the predicted state $x_k^-$ and the anticipated measurement random variable $y_k$ is evaluated as:


$$\text{Cov}(x_k^-, y^{-}_k) = \text{Cov}(x_k^-, H_k x_k^- + z_k) = P_k^- H_k^T.$$

Therefore, the joint distribution of the predicted state and the upcoming measurement is a block-structured multivariate Gaussian:
$$
\begin{bmatrix} x_k^- \\ y^{-}_k \end{bmatrix} \sim \mathscr{N}\left(
\begin{bmatrix} m_k^- \\ H_k m_k^- \end{bmatrix},
\begin{bmatrix} P_k^- & P_k^- H_k^T \\ H_k P_k^- & H_k P_k^- H_k^T + \Sigma_m \end{bmatrix}
\right).
$$

---

**Measurement Update (Correction Step):**

At time step $k$, a physical measurement is observed, causing the random variable to take a specific numerical realization: $y_k = y^{\mathrm{obs}}_{k}$.

Applying the standard conditioning properties of multivariate normal distributions, we update our prior belief of the state by slicing the joint distribution at the realization $y^{\mathrm{obs}}_{k}$. This yields the posterior state distribution:


$$x^{+}_k\triangleq (x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k}) \sim \mathscr{N}(m_k, P_k),$$


where the updated mean $m_k$ and updated covariance $P_k$ are given by:
\begin{align*}
K_k &\triangleq P_k^- H_k^T (H_k P_k^- H_k^T + \Sigma_m)^{-1}, \\
m_k &= m_k^- + K_k (y^{\mathrm{obs}}_{k} - H_k m_k^-), \\
P_k &= (I - K_k H_k) P_k^-.
\end{align*}

Here, $K_k$ is the Kalman Gain, and
$$ m_k=\mathbb{E}[x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k}]$$ acts as the updated state estimate, and
$$P_k=\text{Var}(x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k})$$ is the new posterior error covariance matrix used to seed the next recursive time-step.

Note that $x^{+}_k$ denotes an abstract random variable distributed according to the conditional law of $x^{-}_k$ given the observed value $y^{\mathrm{obs}}_{k}$.

You may refer to this note on [multivariate Gaussians](https://github.com/mugalan/introduction-to-statistical-learning/blob/main/Multivariate_Gaussian_Distributions.ipynb) for the exact details of extracting the mean and the covariance of the conditional distribution.

---

**Error Dynamics**

For clarity, define the actual estimation error by
$$
\varepsilon_k \triangleq x_k - m_k,
\qquad
\varepsilon_k^- \triangleq x_k - m_k^- .
$$

From the measurement update,
$$
m_k = m_k^- + K_k\left(y_k^{\mathrm{obs}} - H_km_k^-\right),
$$
and using
$$
y_k^{\mathrm{obs}} = H_kx_k + z_k,
$$
we obtain
$$
\begin{aligned}
\varepsilon_k
&= x_k - m_k \\
&= x_k - m_k^- - K_k\left(H_kx_k + z_k - H_km_k^-\right) \\
&= \left(I-K_kH_k\right)(x_k-m_k^-) - K_kz_k \\
&= \left(I-K_kH_k\right)\varepsilon_k^- - K_kz_k .
\end{aligned}
$$

Moreover, from the prediction step,
$$
x_k = A_{k-1}x_{k-1} + G_{k-1}w_{k-1},
\qquad
m_k^- = A_{k-1}m_{k-1},
$$
we have
$$
\varepsilon_k^-
=
A_{k-1}\varepsilon_{k-1}
+
G_{k-1}w_{k-1}.
$$

Therefore,
$$
\boxed{
\varepsilon_k
=
\left(I-K_kH_k\right)A_{k-1}\varepsilon_{k-1}
+
\left(I-K_kH_k\right)G_{k-1}w_{k-1}
-
K_kz_k .
}
$$

## 1-D Example

Consider the scalar linear-Gaussian filter model:
\begin{aligned}
x^-_k &= a\,x^+_{k-1} + w_{k-1},\qquad w_{k-1}\sim\mathscr N(0,\Sigma_q),\\
y^-_k &= h\,x^-_k + z_k,\qquad\;\;\;\; z_k\sim\mathscr N(0,\Sigma_r),
\end{aligned}
where
$$x^{+}_{k-1} \sim \mathscr{N}(m_{k-1}, P_{k-1}).$$


**Prediction:**

Let $$x_k^- \sim \mathscr{N}(m_k^-, P_k^-).$$ From the above filter model we have:
\begin{aligned}
m_k^- &= a\,m_{k-1},\\
P_k^- &= a^2 P_{k-1} + \Sigma_q.
\end{aligned}

**Predicted Measurment:**

From the filter model we have that
$$y_k^- \sim \mathscr{N}(hm_k^-, h^2P_k^-+\Sigma_r).$$


Thus
$$
\begin{bmatrix} x_k^- \\ y^{-}_k \end{bmatrix} \sim \mathscr{N}\left(
\begin{bmatrix} m_k^- \\ h m_k^- \end{bmatrix},
\begin{bmatrix} P_k^- & P_k^- h \\ P_k^-h & h^2P_k^-+\Sigma_r \end{bmatrix}
\right).
$$



**Measurement Update (Correction Step):**

At time step $k$, a physical measurement is observed, causing the random variable to take a specific numerical realization: $y_k = y^{\mathrm{obs}}_{k}$.

Applying the standard conditioning properties of multivariate normal distributions, we update our prior belief of the state by slicing the joint distribution at the realization $y^{\mathrm{obs}}_{k}$. This yields the posterior state distribution:


$$x^{+}_k\triangleq (x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k}) \sim \mathscr{N}(m_k, P_k),$$

where the updated mean $m_k$ and updated covariance $P_k$ are given by:
\begin{align*}
K_k &\triangleq \frac{P_k^- h }{(h^2P_k^- + \Sigma_r)}, \\
m_k &= m_k^- + K_k (y^{\mathrm{obs}}_{k} - h m_k^-), \\
P_k &= (1 - K_k h)\,P_k^-
= \Bigl(1 - \frac{P_k^- h^2}{(h^2P_k^- + \Sigma_r)}\Bigr) P_k^- .
\end{align*}

Here, $K_k$ is the Kalman Gain, and
$$ m_k=\mathbb{E}[x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k}]$$ acts as the updated state estimate, and
$$P_k=\text{Var}(x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k})$$ is the new posterior error covariance matrix used to seed the next recursive time-step.

Note that $x^{+}_k$ denotes an abstract random variable distributed according to the conditional law of $x^{-}_k$ given the observed value $y^{\mathrm{obs}}_{k}$.



In [ ]:
import numpy as np

def make_cv1d(dt: float = 0.1,
              q: float = 1e-2,
              r: float = 1e-1,
              x0=(0.0, 1.0),
              seed: int | None = None) -> sims.LinearGaussianSystemSyms:
    """
    Build a 1D constant-velocity linear Gaussian system:

        x_k = A x_{k-1} + w_{k-1},      w ~ N(0, Q)
        y_k = H x_k       + z_k,        z ~ N(0, R)

    State: x = [position, velocity]^T  (n=2)
    Measurement: y = position (scalar, p=1)

    Parameters
    ----------
    dt : float
        Sampling period Δt.
    q : float
        Continuous white-acceleration noise intensity (process noise scale).
        Discrete-time Q = q * [[dt^3/3, dt^2/2],
                               [dt^2/2, dt     ]].
    r : float
        Measurement noise std. R = [[r^2]] (scalar variance).
    x0 : tuple[float, float]
        Initial state (position, velocity).
    seed : int | None
        Seed for reproducible randomness.

    Returns
    -------
    LinearGaussianSystemSyms
        System with A, H, Sigma_p (Q), Sigma_m (R), and initial state x0.
    """
    A = np.array([[1.0, dt],
                  [0.0, 1.0]], dtype=float)

    # Measure position only (scalar)
    H = np.array([[1.0, 0.0]], dtype=float)  # shape (1,2)

    # Discrete CV process noise covariance (from white-acceleration model)
    Q = q * np.array([[dt**3/3.0, dt**2/2.0],
                      [dt**2/2.0, dt      ]], dtype=float)

    # Measurement noise covariance (scalar)
    R = np.array([[r**2]], dtype=float)

    x0 = np.asarray(x0, dtype=float)
    if x0.shape != (2,):
        raise ValueError(f"`x0` must be shape (2,), got {x0.shape}.")

    rng = np.random.default_rng(seed)
    return sims.LinearGaussianSystemSyms(A=A, H=H, Sigma_p=Q, Sigma_m=R, x0=x0, rng=rng)

# Must have p=1 in your system (scalar measurement). n can be >1.
sys = make_cv1d(dt=0.1, q=5e-2, r=1.0, x0=(0.0, 0.5), seed=7)  # p=1
# Run with simulated Y (T provided)
sys.animate_measurement_gaussians_scalar(T=80, m0=np.zeros(sys.n), P0=np.eye(sys.n)*100,
                                         frame_ms=120, save_html_path=None, show=True)

# Or if you've already collected Y (shape (T,) or (T,1)):
# _, Y = sys.simulate(T=100)
# sys.animate_measurement_gaussians_scalar(Y=Y, m0=np.zeros(sys.n), P0=np.eye(sys.n)*100,
#                                          save_html_path="kf_scalar_y_gaussians.html", auto_play=False)

## Simulation 2D-example

We consider a two-dimensional constant-velocity dynamical system. The hidden state at time step $k$ is

$$
x_k =
\begin{bmatrix}
p_x(k)\\
p_y(k)\\
v_x(k)\\
v_y(k)
\end{bmatrix},
$$

where $(p_x(k),p_y(k))$ denote the position components and $(v_x(k),v_y(k))$ denote the velocity components.

The measurement consists only of the two position components:

$$
y_k =
\begin{bmatrix}
p_x^{\mathrm{meas}}(k)\\
p_y^{\mathrm{meas}}(k)
\end{bmatrix}.
$$

The linear Gaussian state-space model is

$$
x_k = A x_{k-1} + G w_{k-1},
$$

$$
y_k = Hx_k + z_k,
$$

where

$$
w_{k-1} \sim \mathscr{N}(0,\Sigma_p),
\qquad
z_k \sim \mathscr{N}(0,\Sigma_m).
$$

The process noise sequence $w_k$, measurement noise sequence $z_k$, and the initial state are assumed mutually independent.

---


Assuming a sampling interval $\Delta t$, the constant-velocity kinematic equations are

$$
p_x(k) = p_x(k-1) + \Delta t\,v_x(k-1),
$$

$$
p_y(k) = p_y(k-1) + \Delta t\,v_y(k-1),
$$

$$
v_x(k) = v_x(k-1),
$$

$$
v_y(k) = v_y(k-1).
$$

Therefore, in matrix form,

$$
x_k =
\begin{bmatrix}
1 & 0 & \Delta t & 0\\
0 & 1 & 0 & \Delta t\\
0 & 0 & 1 & 0\\
0 & 0 & 0 & 1
\end{bmatrix}
x_{k-1}
+
G w_{k-1}.
$$

Thus,

$$
A =
\begin{bmatrix}
1 & 0 & \Delta t & 0\\
0 & 1 & 0 & \Delta t\\
0 & 0 & 1 & 0\\
0 & 0 & 0 & 1
\end{bmatrix}.
$$

---

Since the measurement contains only the position components, we have

$$
y_k =
\begin{bmatrix}
p_x(k)\\
p_y(k)
\end{bmatrix}
+
z_k.
$$

Equivalently,

$$
y_k = Hx_k + z_k,
$$

where

$$
H =
\begin{bmatrix}
1 & 0 & 0 & 0\\
0 & 1 & 0 & 0
\end{bmatrix}.
$$

The measurement noise is modeled as

$$
z_k \sim \mathscr{N}(0,\Sigma_m),
$$

with

$$
\Sigma_m = r^2 I_2.
$$

Here, $r$ is the standard deviation of the measurement noise in each position coordinate.

---


A common way to model uncertainty in a constant-velocity system is to assume that the unmodeled acceleration is random. Let

$$
w_{k-1} =
\begin{bmatrix}
a_x(k-1)\\
a_y(k-1)
\end{bmatrix},
$$

where

$$
w_{k-1} \sim \mathscr{N}(0,q^2I_2).
$$

Here, $q$ controls the acceleration-noise intensity.

Over one sampling interval $\Delta t$, the random acceleration affects both position and velocity:

$$
p_x(k)
=
p_x(k-1)
+
\Delta t\,v_x(k-1)
+
\frac{1}{2}\Delta t^2 a_x(k-1),
$$

$$
p_y(k)
=
p_y(k-1)
+
\Delta t\,v_y(k-1)
+
\frac{1}{2}\Delta t^2 a_y(k-1),
$$

$$
v_x(k)
=
v_x(k-1)
+
\Delta t\,a_x(k-1),
$$

$$
v_y(k)
=
v_y(k-1)
+
\Delta t\,a_y(k-1).
$$

Therefore,

$$
G =
\begin{bmatrix}
\frac{1}{2}\Delta t^2 & 0\\
0 & \frac{1}{2}\Delta t^2\\
\Delta t & 0\\
0 & \Delta t
\end{bmatrix},
$$

and

$$
\Sigma_p = q I_2.
$$

The induced state-space process covariance is

$$
Q
=
G\Sigma_pG^T.
$$

Since $\Sigma_p=qI_2$, this becomes

$$
Q
=
qGG^T.
$$

Explicitly,

$$
Q
=
q
\begin{bmatrix}
\frac{1}{4}\Delta t^4 & 0 & \frac{1}{2}\Delta t^3 & 0\\
0 & \frac{1}{4}\Delta t^4 & 0 & \frac{1}{2}\Delta t^3\\
\frac{1}{2}\Delta t^3 & 0 & \Delta t^2 & 0\\
0 & \frac{1}{2}\Delta t^3 & 0 & \Delta t^2
\end{bmatrix}.
$$

---

Thus, the full Gaussian filter is

$$
x^-_k =
Ax^+_{k-1}+Gw_{k-1},
\qquad
w_{k-1}\sim\mathscr{N}(0,q^2I_2),
$$

$$
y^-_k =
Hx^-_k+z_k,
\qquad
z_k\sim\mathscr{N}(0,r^2I_2).
$$

In [ ]:
# 2D position-only measurements (p=2), CV model in x & y
def make_cv2d(
    dt=0.1,
    q=1e-4,
    r=0.5,
    x0=(0, 0, 1, 0.5),
    seed=0,
    *,
    use_G=True,
    noise_model="accel_white",
):
    """
    Build a 2D constant-velocity (CV) linear-Gaussian system.

    States: x = [x, y, vx, vy]^T
      A = [[1, 0, dt, 0 ],
           [0, 1, 0 , dt],
           [0, 0, 1 , 0 ],
           [0, 0, 0 , 1 ]]

    Measurements: y = [x, y]^T  (position only)

    Process-noise options
    ---------------------
    - use_G=True, noise_model='accel_white'  (default):
        Uses an explicit G that maps a 2D white acceleration noise (w ~ N(0, q I_2))
        into the state:
            G = [[dt^2/2,     0   ],
                 [   0  ,  dt^2/2],
                 [  dt ,     0   ],
                 [   0 ,    dt   ]]
        Sigma_p = q * I_2   (in w-space)
        → State-space covariance is G Sigma_p G^T
        (This is the common “white-acceleration” CV model.)

    - use_G=False:
        Legacy behavior (no G). We provide the classic state-space Q directly
        corresponding to (velocity random-walk discretization):
            Q_block = [[dt^3/3, dt^2/2],
                       [dt^2/2, dt     ]] * q
        Q = blockdiag(Q_block, Q_block)
        Sigma_p = Q  (already in state space), G=None

    Notes
    -----
    The two variants imply slightly different discrete-time process covariances.
    Pick the one that matches your physical assumption / reference text.
    """
    A = np.array([
        [1, 0, dt,  0],
        [0, 1,  0, dt],
        [0, 0,  1,  0],
        [0, 0,  0,  1],
    ])
    H = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
    ])
    R = np.eye(2) * (r**2)

    if use_G:
        if noise_model != "accel_white":
            raise ValueError("When use_G=True, supported noise_model is only 'accel_white'.")
        # White acceleration injected into vx, vy
        G = np.array([
            [0.5*dt*dt, 0.0       ],
            [0.0      , 0.5*dt*dt],
            [dt       , 0.0       ],
            [0.0      , dt        ],
        ])
        Sigma_p = np.eye(2) * (q**2)  # w-space covariance (2x2)
        return sims.LinearGaussianSystemSyms(
            A=A, H=H, Sigma_p=Sigma_p, Sigma_m=R,
            x0=np.asarray(x0, float),
            rng=np.random.default_rng(seed),
            G=G
        )
    else:
        # Legacy/state-space Q (velocity random-walk discretization)
        Q_block = np.array([[dt**3/3, dt**2/2],
                            [dt**2/2, dt      ]], dtype=float) * q
        Q = np.block([
            [Q_block,               np.zeros((2, 2))],
            [np.zeros((2, 2)),      Q_block        ],
        ])
        return sims.LinearGaussianSystemSyms(
            A=A, H=H, Sigma_p=Q, Sigma_m=R,
            x0=np.asarray(x0, float),
            rng=np.random.default_rng(seed),
            G=G
        )

In [ ]:
sys = make_cv2d(dt=0.1, q=1e-2, r=0.5, x0=(0,0, 0.5, -0.2), seed=7)
X2, Y2 = sys.simulate(T=300)
sys.plot_y(Y2, nbins=40, component_labels=["pos_x", "pos_y"])

In [ ]:
# Option A: simulate 300 steps internally, starting from broad prior
M, Yhat= sys.filter_with_kf_and_plot(T=300, Y=Y2, m0=np.array([0,0, 0,0]), P0=np.eye(4)*100,
                                      component_labels=["pos_x","pos_y"], show=True)

# Option B: if you already have measurements Y, just pass them:
# X, Y = sys.simulate(T=300)
# M, Yhat = sys.filter_with_kf_and_plot(Y=Y, m0=np.array([0,0, 0,0]), P0=np.eye(4)*100,
#                                       component_labels=["pos_x","pos_y"])